Perfect ✅ — below is your **complete, fixed, and fully integrated codebase** for a **multi-agent router system**.
This setup includes:

* 🧭 A **Router Agent** that decides which sub-agent to call.
* 🤖 **Child Agents** for math, pollution, and weather tasks.
* ⚙️ A dynamic agent factory to create and manage them.
* 🧩 Working MCP servers (math, pollution, weather).

---

## 📁 Project Structure

```
project/
│
├── main.py
├── agent_factory/
│   └── dynamic_agent_factory.py
├── mcp_servers/
│   ├── math_server.py
│   ├── pollution-mcp_server.py
│   └── weather-mcp_server.py
├── mcp_clients/
│   └── universal_mcp_client.py
└── config/
    └── settings.py
```

---

## 🧭 `main.py`

```python
import asyncio
from agent_factory.dynamic_agent_factory import DynamicAgentFactory
from config.settings import AGENT_CONFIG

async def main():
    factory = DynamicAgentFactory(AGENT_CONFIG)

    print("\n🤖 Multi-Agent Router System Started")
    print("Agents available:", ", ".join(AGENT_CONFIG.keys()))
    print("Type 'exit' to quit.\n")

    router_agent = await factory.get_agent("router_agent")

    while True:
        query = input("[User] > ").strip()
        if query.lower() in ["exit", "quit"]:
            print("👋 Exiting. Goodbye!")
            break

        try:
            print("\n🧭 Router deciding which agent to use...\n")
            route_decision = await router_agent.ainvoke({"input": query})
            selected_agent_name = route_decision["output"].strip()

            if selected_agent_name not in AGENT_CONFIG:
                print(f"⚠️ Router selected invalid agent: {selected_agent_name}")
                continue

            print(f"➡️ Routing query to: {selected_agent_name}")
            selected_agent = await factory.get_agent(selected_agent_name)

            print("\n🤔 Thinking...\n")
            response = await selected_agent.ainvoke({"input": query})
            print(f"🧠 Agent Response:\n{response['output']}\n")

        except Exception as e:
            print(f"⚠️ Unexpected error: {e}")
            import traceback
            traceback.print_exc()

if __name__ == "__main__":
    asyncio.run(main())
```

---

## ⚙️ `agent_factory/dynamic_agent_factory.py`

```python
import importlib
from typing import Dict, Any
from langchain_openai import ChatOpenAI
from langchain.agents import initialize_agent, AgentType
from mcp_clients.universal_mcp_client import load_all_mcp_tools

class DynamicAgentFactory:
    """
    Dynamically creates and manages LangChain agents
    based on configuration in AGENT_CONFIG.
    """

    def __init__(self, config: Dict[str, Any]):
        self.config = config
        self.registry = {}

    async def create_agent(self, name: str):
        if name not in self.config:
            raise ValueError(f"Agent '{name}' not found in configuration.")

        agent_cfg = self.config[name]
        print(f"🔧 Creating agent: {name}")
        print(f"   ↳ LLM: {agent_cfg['llm_model']}")
        print(f"   ↳ MCP Servers: {agent_cfg.get('mcp_servers', [])}")

        llm = ChatOpenAI(model=agent_cfg["llm_model"], temperature=0.2)

        tools = []
        try:
            tools = await load_all_mcp_tools(agent_cfg)
        except Exception as e:
            print(f"⚠️ Error loading MCP tools for {name}: {e}")

        agent = initialize_agent(
            tools=tools,
            llm=llm,
            agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
            verbose=False,
            handle_parsing_errors=True
        )

        self.registry[name] = {
            "llm": llm,
            "tools": tools,
            "agent": agent,
        }

        return agent

    async def get_agent(self, name: str):
        if name not in self.registry:
            await self.create_agent(name)
        return self.registry[name]["agent"]
```

---

## 🌐 `mcp_clients/universal_mcp_client.py`

```python
import os
from langchain_mcp_adapters.client import MultiServerMCPClient

async def load_all_mcp_tools(agent_cfg: dict):
    """Dynamically load all MCP tools based on config."""
    current_dir = os.path.dirname(os.path.abspath(__file__))
    servers_config = {}

    for mcp_name in agent_cfg.get("mcp_servers", []):
        server_file = f"{mcp_name}.py"
        server_path = os.path.join(current_dir, "..", "mcp_servers", server_file)
        server_path = os.path.abspath(server_path)

        if not os.path.exists(server_path):
            print(f"⚠️ Warning: MCP server file not found at {server_path}")
            continue

        servers_config[mcp_name] = {
            "command": "python",
            "args": [server_path],
            "transport": "stdio",
        }

    if not servers_config:
        return []

    client = MultiServerMCPClient(servers_config)
    all_tools = await client.get_tools()

    print(f"🔧 Loaded {len(all_tools)} tools from: {', '.join(servers_config.keys())}")
    return all_tools
```

---

## 🧠 `config/settings.py`

```python
import os
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
DEFAULT_MODEL = "gpt-4o"

AGENT_CONFIG = {
    # Math + Pollution Agent
    "agent_math_env": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": (
            "You are a helpful assistant specializing in mathematics and environmental analysis. "
            "Use the available math and pollution MCP tools effectively. "
            "Always provide the final answer clearly, starting with 'Final Answer:'."
        ),
        "mcp_servers": ["math_server", "pollution-mcp_server"],
    },

    # Weather Agent
    "agent_weather": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": (
            "You are a friendly assistant specializing in weather and environmental insights. "
            "Use the weather MCP tools to provide accurate and concise information. "
            "Always start your main conclusion with 'Final Answer:'."
        ),
        "mcp_servers": ["weather-mcp_server"],
    },

    # Router Agent
    "router_agent": {
        "llm": "openai",
        "llm_model": "gpt-4o-mini",
        "system_prompt": (
            "You are a routing assistant. Based on the user query, decide which specialized agent to use.\n\n"
            "Routing Rules:\n"
            "- If the query involves numbers, calculations, or math → use 'agent_math_env'\n"
            "- If it involves AQI, pollution, or air quality → use 'agent_math_env'\n"
            "- If it involves weather, temperature, or forecast → use 'agent_weather'\n"
            "Return only the agent name (e.g., 'agent_math_env' or 'agent_weather')."
        ),
        "mcp_servers": []
    },
}
```

---

## ➗ `mcp_servers/math_server.py`

```python
from fastmcp import FastMCP
from typing import List, Union

mcp = FastMCP("Math Server")

@mcp.tool()
def add(numbers: List[Union[int, float]]) -> Union[int, float]:
    """Add multiple numbers together."""
    return sum(numbers)

@mcp.tool()
def multiply(numbers: List[Union[int, float]]) -> Union[int, float]:
    """Multiply multiple numbers together."""
    result = 1
    for num in numbers:
        result *= num
    return result

@mcp.tool()
def subtract(numbers: List[Union[int, float]]) -> Union[int, float]:
    """Subtract numbers sequentially."""
    if not numbers:
        return 0
    result = numbers[0]
    for num in numbers[1:]:
        result -= num
    return result

@mcp.tool()
def divide(numbers: List[Union[int, float]]) -> Union[int, float]:
    """Divide numbers sequentially."""
    if not numbers:
        return 0
    result = numbers[0]
    for num in numbers[1:]:
        if num == 0:
            raise ValueError("Cannot divide by zero")
        result /= num
    return result

if __name__ == "__main__":
    print("Starting Math Server...")
    mcp.run(transport="stdio")
```

---

## 🌍 `mcp_servers/pollution-mcp_server.py`

```python
from fastmcp import FastMCP

POLLUTION_DATA = {
    "Delhi": "AQI 320 (Very Poor)",
    "Mumbai": "AQI 160 (Moderate)",
    "Paris": "AQI 70 (Good)"
}

mcp = FastMCP("pollution-mcp")

@mcp.tool()
def get_pollution(location: str) -> str:
    """Return pollution info for a given location."""
    return POLLUTION_DATA.get(location.title(), "No data available.")

if __name__ == "__main__":
    print("🌍 Starting Pollution MCP Server...")
    mcp.run(transport="stdio")
```

---

## 🌦️ `mcp_servers/weather-mcp_server.py`

```python
from fastmcp import FastMCP

WEATHER_DATA = {
    "Delhi": "☀️ 35°C, Clear Sky",
    "Mumbai": "🌧️ 29°C, Light Rain",
    "Paris": "⛅ 22°C, Partly Cloudy",
    "New York": "🌤️ 18°C, Breezy",
    "Tokyo": "🌧️ 24°C, Showers"
}

mcp = FastMCP("weather-mcp")

@mcp.tool()
def get_weather(city: str) -> str:
    """Get current weather information for a given city."""
    result = WEATHER_DATA.get(city.title(), "No weather data available.")
    return f"Weather in {city.title()}: {result}"

@mcp.tool()
def list_weather_cities() -> str:
    """List all available cities for weather information."""
    return "Available cities: " + ", ".join(WEATHER_DATA.keys())

if __name__ == "__main__":
    print("🌦️ Starting Weather MCP Server...")
    mcp.run(transport="stdio")
```

---

## ✅ How to Run

1. Set your OpenAI key in `.env`:

   ```
   OPENAI_API_KEY=your_openai_api_key_here
   ```

2. Start the system:

   ```bash
   python main.py
   ```

3. Try queries:

   ```
   [User] > add 5 and 10
   [User] > what's the weather in Paris?
   [User] > what is AQI in Delhi?
   ```

---

## 💡 Outcome

* Router automatically picks the correct agent
* Math & Pollution handled by `agent_math_env`
* Weather handled by `agent_weather`
* You can easily add new agents (e.g., finance, calendar) later

---

Would you like me to extend this router to **explain its reasoning** (e.g., “I chose `agent_weather` because your question mentions weather”) before delegating? That’s a great next step for debugging and transparency.
